In [ ]:
pip install z3-solver

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.5/29.5 MB 45.4 MB/s eta 0:00:00


In [ ]:
from z3 import *
from itertools import combinations

In [ ]:
def at_least_one(bool_vars):
    return Or(bool_vars)

def at_most_one(bool_vars):
    return [Not(And(pair[0], pair[1])) for pair in combinations(bool_vars, 2)]

def at_most_k(bool_vars, k):
  return [Not]

def exactly_one(bool_vars):
    return at_most_one(bool_vars) + [at_least_one(bool_vars)]

In [ ]:
def print_schedule(n, model, v):
    weeks = n - 1
    periods = n // 2
    schedule = [["" for _ in range(weeks)] for _ in range(periods)]

    for i in range(weeks):
        for j in range(periods):
            home = away = None
            for t in range(n):
                if is_true(model.evaluate(v[i][j][0][t])):
                    home = t + 1
                if is_true(model.evaluate(v[i][j][1][t])):
                    away = t + 1
            if home is not None and away is not None:
                schedule[j][i] = f"{home} ∨ {away}"

    # Print the results
    header = "          " + "".join([f"Week {i+1}   " for i in range(weeks)])
    print(header)
    for p in range(periods):
        row = f"Period {p+1}  " + "".join([f"{schedule[p][w]}    " for w in range(weeks)])
        print(row)


In [ ]:
def STS_SAT(n):
  # All the variables we need: for each game's week i, for each period j, for each home-away m,
  # n variables that determine which team must be assigned.
  v = [[[[Bool(f"v_{i}_{j}_{m}_{t}") for t in range(n)] for m in range(2)] for j in range (n//2)] for i in range(n-1)]

  s = Solver()

  # A cell has only one value
  for i in range(n-1):
    for j in range(n//2):
      for m in range(2):
        s.add(exactly_one(v[i][j][m]))

  # A team plays only once in a week
  for i in range(n-1):
      for t in range(n):
          occurrences = []
          for j in range(n // 2):
              for m in range(2):
                  occurrences.append(v[i][j][m][t])
          s.add(AtMost(*occurrences, 1))

  # Every teams plays again each other only once
  for t1 in range(n):
    for t2 in range(t1 + 1, n):
        matches = []
        for i in range(n - 1):
            for j in range(n // 2):
                match = And(
                    Or(And(v[i][j][0][t1], v[i][j][1][t2]), And(v[i][j][0][t2], v[i][j][1][t1]))
                )
                matches.append(match)
        s.add(AtMost(*matches, 1))

  # Every team plays in the same period at most 2 times
  for t in range(n):
        for j in range(n // 2):
            appearances = []
            for i in range(n - 1):
                for m in range(2):
                    appearances.append(v[i][j][m][t])
            s.add(AtMost(*appearances, 2))


  # Define a model solution
  s.add(v[0][0][0][0]==True)
  c = 1
  for i in range(1, n-1):
    if i%2!=0:
      s.add(v[i][c][0][0]==True)
      if i!=n-1:
        s.add(v[i+1][c][0][0]==True)
        c += 1
    else:
      continue

  if s.check() == sat:
    m = s.model()

    return m, v
  else:
      print("Failed to solve")





In [ ]:
result = STS_SAT(12)

if result:
    model, variables = result
    print_schedule(12, model, variables)
else:
    print("Solver failed, cannot print schedule.")

          Week 1   Week 2   Week 3   Week 4   Week 5   Week 6   Week 7   Week 8   Week 9   Week 10   Week 11   
Period 1  1 ∨ 9    11 ∨ 2    12 ∨ 10    5 ∨ 4    6 ∨ 4    8 ∨ 9    12 ∨ 7    7 ∨ 6    11 ∨ 8    2 ∨ 10    3 ∨ 5    
Period 2  10 ∨ 5    1 ∨ 6    1 ∨ 11    2 ∨ 7    8 ∨ 12    11 ∨ 3    6 ∨ 3    12 ∨ 2    5 ∨ 7    8 ∨ 4    9 ∨ 10    
Period 3  7 ∨ 8    3 ∨ 9    5 ∨ 6    1 ∨ 8    1 ∨ 5    6 ∨ 12    10 ∨ 4    11 ∨ 4    2 ∨ 9    3 ∨ 12    11 ∨ 7    
Period 4  6 ∨ 2    10 ∨ 8    7 ∨ 3    10 ∨ 3    7 ∨ 9    1 ∨ 4    1 ∨ 2    5 ∨ 9    4 ∨ 12    5 ∨ 11    6 ∨ 8    
Period 5  11 ∨ 12    5 ∨ 12    9 ∨ 4    11 ∨ 6    2 ∨ 3    7 ∨ 10    5 ∨ 8    1 ∨ 10    1 ∨ 3    6 ∨ 9    4 ∨ 2    
Period 6  3 ∨ 4    7 ∨ 4    2 ∨ 8    12 ∨ 9    11 ∨ 10    2 ∨ 5    9 ∨ 11    3 ∨ 8    6 ∨ 10    1 ∨ 7    1 ∨ 12    
